In [13]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [14]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}
@tool
def get_product(name: str) -> str:
   
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [15]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [16]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [20]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [21]:
ask("what is the price and reviews of smart watch")

The price of the smart watch is $199.99 and it has a rating of 4.3 out of 5. It tracks heart rate and sleep, has a 5-day battery life, and is water-resistant.


In [25]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}

    result = agent2.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config
    )

    print(result["messages"][-1].content)

In [26]:
ask2("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [27]:
ask2("other details about this product.")

The wireless headphones have a rating of 4.6 and a description of "Over-ear Bluetooth, 30-hr battery, active noise cancellation." They are also in stock.
